# PII Column Classifier on Colab

This is a notebook that you can use to run the PII Column Classifier pipeline in Google Colab. 

**Setup once:** Runtime -> Change runtime type -> **T4 GPU**. 

**Colab Secrets** (key icon) - add with "Notebook access" on:
- `ANTHROPIC_API_KEY` (only needed to generate synthetic data)
- `HF_TOKEN` (optional - only to upload the fine-tuned model to Hugging Face Hub)

**Note:** This notebook uses a **public GitHub repo** - no GitHub token needed!

Code comes from GitHub; data/models/results are stored on Google Drive so they survive disconnects.

In [ ]:
# ---- EDIT THESE ----
GITHUB_REPO = "username/pii-column-classifier"   # Public repo: owner/name
BRANCH = "main"
HF_MODEL_REPO = "your-username/pii-column-bge-small"  # (Optional) Where to push fine-tuned model
RUN = "colab_v1"                                       # Change for each experiment

In [ ]:
import os
from google.colab import userdata

# Use to fix Google mount issues so use local storage instead of Drive (data will be lost on disconnect!)
os.environ['PIPELINE_BASE_DIR'] = '/content/pii_artifacts'
os.makedirs(os.environ['PIPELINE_BASE_DIR'], exist_ok=True)

print("⚠️  WARNING: Using local storage - data will be lost if runtime disconnects!")

# Clone public repo (no authentication needed)
if not os.path.exists('/content/pipeline/.git'):
    !git clone -q -b {BRANCH} https://github.com/{GITHUB_REPO}.git /content/pipeline
%cd /content/pipeline
!git pull -q origin {BRANCH}
!git log --oneline -3

In [ ]:
!pip install -q --upgrade anthropic>=0.40
!pip install -q -r requirements.txt
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. (Optional) Generate synthetic data
Skipped if the CSV already exists on Drive. Add `--mock` to test the plumbing without the API.

In [ ]:
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
csv = os.environ['PIPELINE_BASE_DIR'] + '/data/synthetic_pii_columns.csv'
if not os.path.exists(csv):
    !python src/synthetic_data.py --total 10000
else:
    print('Synthetic data already present:', csv)

## 2. Train + compare (TF-IDF vs stock BGE vs fine-tuned BGE)

Trains 11 configurations:
- TF-IDF (char/word/combined) × LinearSVC/XGBoost = 6 configs
- Stock BGE-Small × LinearSVC/XGBoost = 2 configs
- Fine-tuned BGE-Small × LinearSVC/XGBoost/ClassifierHead = 3 configs

In [ ]:
!python src/train_models.py --run-name {RUN} --finetune-epochs 8 --finetune-batch-size 64 --minilm-batch-size 256

## 3. Checkpoints + probabilities

In [ ]:
!python src/evaluate_checkpoints.py --run-name {RUN} --minilm-batch-size 256
!python src/evaluate_probabilities.py --run-name {RUN} --minilm-batch-size 256

In [ ]:
import pandas as pd
base = os.environ['PIPELINE_BASE_DIR']
pd.read_csv(f'{base}/results/{RUN}/accuracy_comparison_all_configurations.csv')

## 4. (Optional) Push the fine-tuned model to the Hugging Face Hub (private)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=userdata.get('HF_TOKEN'))
api.create_repo(HF_MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=f"{base}/models/{RUN}/ft_final", repo_id=HF_MODEL_REPO)
print("Uploaded to", HF_MODEL_REPO)